# Bankruptcy Prediction Using Machine Learning

## Aim

To build a machine learning pipeline that can predict whether a company is likely to go bankrupt (`failed`) or remain active (`alive`) based on its financial indicators.

## Objectives

1. Understand the dataset properly.
2. Perform beginner-friendly Exploratory Data Analysis (EDA).
3. Clean and prepare the data for modeling.
4. Build multiple machine learning models.
5. Compare model performance using suitable evaluation metrics.
6. Save the best model so it can later be used in a UI or application.

## Problem Statement

We have yearly financial records of companies in the United States.  
Using these values, we want to train a model that can classify a company as:

- `0` -> Alive
- `1` -> Failed / Bankrupt

This notebook is written in a simple, lab-style format so that each step is easy to follow.

## Workflow Used In This Notebook

1. Import libraries
2. Load the dataset
3. Understand the dataset structure
4. Perform EDA
5. Handle data cleaning and preprocessing
6. Split the data carefully
7. Train multiple models
8. Evaluate and compare the models
9. Tune the final decision threshold
10. Save the final model for future use

In [ ]:
# If required, uncomment the next line and install the dependencies.
# %pip install pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_PATH = Path("american_bankruptcy.csv")
TARGET_COL = "status_label"
GROUP_COL = "company_name"
TIME_COL = "year"

## Step 1: Load the Dataset

We first load the CSV file and inspect a few rows.

In [ ]:
df = pd.read_csv(DATA_PATH)

# The dataset contains a hidden BOM character in the first column name.
df = df.rename(columns={"\ufeffcompany_name": "company_name"})

print("Dataset shape:", df.shape)
df.head()

## Step 2: Basic Dataset Understanding

In this section, we check:

- number of rows and columns
- column names
- datatypes
- missing values
- how the target variable is distributed

In [ ]:
print("Columns in the dataset:")
print(list(df.columns))

print("\nDatatype information:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_count"))

In [ ]:
target_distribution = (
    df[TARGET_COL]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="count")
)
target_distribution["percentage"] = (target_distribution["count"] / len(df) * 100).round(2)
target_distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COL, palette="Set2")
plt.title("Class Distribution")
plt.xlabel("Status Label")
plt.ylabel("Count")
plt.show()

### Observation

The dataset is imbalanced because `alive` companies are much more common than `failed` companies.  
This means accuracy alone is not enough. We also need metrics such as:

- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC

## Step 3: Check Company-Level Data Pattern

The same company appears in multiple years.  
This is important because if the same company appears in both training and testing data, the model may get an unfair advantage.

In [ ]:
company_summary = pd.DataFrame({
    "rows_per_company": df.groupby(GROUP_COL).size(),
    "unique_labels_per_company": df.groupby(GROUP_COL)[TARGET_COL].nunique(),
})

print("Total unique companies:", df[GROUP_COL].nunique())
print("Minimum rows per company:", company_summary["rows_per_company"].min())
print("Maximum rows per company:", company_summary["rows_per_company"].max())
print("Companies with more than one label:", (company_summary["unique_labels_per_company"] > 1).sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df[TIME_COL].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Number of Records by Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

df.groupby(GROUP_COL).size().plot(kind="hist", bins=20, ax=axes[1], color="#F58518")
axes[1].set_title("Distribution of Records per Company")
axes[1].set_xlabel("Rows per Company")

plt.tight_layout()
plt.show()

### Observation

Since companies repeat across multiple years, we should not do a normal random row split.  
Instead, we will use **group-aware splitting** based on `company_name`.

## Step 4: Data Cleaning and Target Encoding

Now we prepare the dataset for modeling.

We will:

- standardize the target labels
- convert `alive` and `failed` into 0 and 1
- choose the model input columns
- keep `year` as a feature
- exclude `company_name` from modeling because it is just an identifier

In [ ]:
df[TARGET_COL] = df[TARGET_COL].str.strip().str.lower()
target_map = {"alive": 0, "failed": 1}
df["target"] = df[TARGET_COL].map(target_map)

if df["target"].isna().any():
    raise ValueError("Unexpected target value found in the dataset.")

numeric_features = [col for col in df.columns if col.startswith("X")] + [TIME_COL]

model_df = df[[GROUP_COL, TARGET_COL, "target"] + numeric_features].copy()
model_df.head()

## Step 5: Exploratory Data Analysis (EDA)

In this section we study:

- summary statistics
- feature distributions
- skewness
- feature correlation
- how features behave for alive vs failed companies

In [ ]:
eda_summary = model_df[numeric_features].describe().T
eda_summary

In [ ]:
skew_table = model_df[numeric_features].skew().sort_values(ascending=False).to_frame("skewness")
skew_table.head(10)

In [ ]:
top_skewed_features = skew_table["skewness"].abs().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, feature in zip(axes.flatten(), top_skewed_features):
    sns.histplot(model_df[feature], bins=50, kde=True, ax=ax, color="#54A24B")
    ax.set_title(f"Distribution of {feature}")

plt.tight_layout()
plt.show()

### Observation

Several financial variables are highly skewed and contain extreme values.  
Because of this, it is a good idea to try both:

- linear models
- tree-based models

In [ ]:
correlation_matrix = model_df[numeric_features].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
class_mean_table = (
    model_df.groupby("target")[numeric_features]
    .mean()
    .T
    .rename(columns={0: "alive_mean", 1: "failed_mean"})
)
class_mean_table["difference"] = (class_mean_table["failed_mean"] - class_mean_table["alive_mean"]).abs()
class_mean_table.sort_values("difference", ascending=False).head(10)

In [ ]:
top_class_gap_features = class_mean_table.sort_values("difference", ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sample_df = model_df.sample(min(len(model_df), 12000), random_state=RANDOM_STATE)

for ax, feature in zip(axes.flatten(), top_class_gap_features):
    sns.boxplot(data=sample_df, x="target", y=feature, ax=ax)
    ax.set_title(f"{feature} by Class")
    ax.set_xlabel("Target (0 = alive, 1 = failed)")

plt.tight_layout()
plt.show()

## Step 6: Prepare Features and Split the Data

Here we create:

- `X` -> input features
- `y` -> target
- `groups` -> company names used for safe splitting

We will create:

- training set
- validation set
- test set

In [ ]:
X = model_df[numeric_features].copy()
y = model_df["target"].copy()
groups = model_df[GROUP_COL].copy()

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=groups))

X_train_val = X.iloc[train_val_idx].reset_index(drop=True)
y_train_val = y.iloc[train_val_idx].reset_index(drop=True)
groups_train_val = groups.iloc[train_val_idx].reset_index(drop=True)

X_test = X.iloc[test_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)
groups_test = groups.iloc[test_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss_val.split(X_train_val, y_train_val, groups=groups_train_val))

X_train = X_train_val.iloc[train_idx].reset_index(drop=True)
y_train = y_train_val.iloc[train_idx].reset_index(drop=True)
groups_train = groups_train_val.iloc[train_idx].reset_index(drop=True)

X_val = X_train_val.iloc[val_idx].reset_index(drop=True)
y_val = y_train_val.iloc[val_idx].reset_index(drop=True)
groups_val = groups_train_val.iloc[val_idx].reset_index(drop=True)

split_table = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Rows": [len(X_train), len(X_val), len(X_test)],
    "Unique Companies": [groups_train.nunique(), groups_val.nunique(), groups_test.nunique()],
    "Failed %": [y_train.mean() * 100, y_val.mean() * 100, y_test.mean() * 100],
})
split_table["Failed %"] = split_table["Failed %"].round(2)
split_table

## Step 7: Build the Preprocessing Pipelines

Before training models, we need to define how the data will be handled.

For numeric features we use:

- median imputation for missing values
- standard scaling for logistic regression
- only imputation for tree-based models

In [ ]:
numeric_preprocessor_scaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

numeric_preprocessor_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

## Step 8: Define the Models

We will compare multiple models:

1. Dummy Baseline
2. Logistic Regression
3. Random Forest
4. Extra Trees
5. Histogram Gradient Boosting

Using multiple models is useful because we do not know in advance which one will work best for this bankruptcy dataset.

In [ ]:
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", ColumnTransformer(
            transformers=[("num", numeric_preprocessor_tree, numeric_features)],
            remainder="drop",
        )),
        ("model", DummyClassifier(strategy="prior")),
    ]
)

logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", ColumnTransformer(
            transformers=[("num", numeric_preprocessor_scaled, numeric_features)],
            remainder="drop",
        )),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", ColumnTransformer(
            transformers=[("num", numeric_preprocessor_tree, numeric_features)],
            remainder="drop",
        )),
        ("model", RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]
)

extra_trees_pipeline = Pipeline(
    steps=[
        ("preprocessor", ColumnTransformer(
            transformers=[("num", numeric_preprocessor_tree, numeric_features)],
            remainder="drop",
        )),
        ("model", ExtraTreesClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]
)

hgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", ColumnTransformer(
            transformers=[("num", numeric_preprocessor_tree, numeric_features)],
            remainder="drop",
        )),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=6,
            max_iter=250,
            min_samples_leaf=30,
            random_state=RANDOM_STATE,
        )),
    ]
)

models = {
    "Dummy Baseline": dummy_pipeline,
    "Logistic Regression": logreg_pipeline,
    "Random Forest": rf_pipeline,
    "Extra Trees": extra_trees_pipeline,
    "HistGradientBoosting": hgb_pipeline,
}

## Step 9: Create Evaluation Helper Functions

To keep the notebook clean, we define helper functions for:

- calculating metrics
- getting predicted probabilities
- performing group-aware cross-validation

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.50):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }


def get_positive_probability(model, X_frame):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_frame)[:, 1]
    return model.decision_function(X_frame)


def group_cv_evaluate(model, X_frame, y_series, group_series, n_splits=5):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    fold_results = []

    for fold_number, (train_idx, valid_idx) in enumerate(cv.split(X_frame, y_series, groups=group_series), start=1):
        current_model = clone(model)
        current_model.fit(X_frame.iloc[train_idx], y_series.iloc[train_idx])
        valid_prob = get_positive_probability(current_model, X_frame.iloc[valid_idx])
        metric_row = compute_metrics(y_series.iloc[valid_idx], valid_prob)
        metric_row["fold"] = fold_number
        fold_results.append(metric_row)

    return pd.DataFrame(fold_results)

## Step 10: Cross-Validation on the Training Data

We now evaluate each model using group-aware cross-validation.

This gives us a more reliable idea of model quality before touching the validation and test sets.

In [ ]:
cv_summaries = []
cv_fold_details = {}

for model_name, model in models.items():
    fold_df = group_cv_evaluate(model, X_train, y_train, groups_train)
    cv_fold_details[model_name] = fold_df

    summary_row = fold_df.drop(columns="fold").mean().to_dict()
    summary_row["model"] = model_name
    cv_summaries.append(summary_row)

cv_results = pd.DataFrame(cv_summaries).sort_values(["pr_auc", "roc_auc"], ascending=False)
cv_results

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=cv_results, x="pr_auc", y="model", palette="viridis")
plt.title("Cross-Validation PR-AUC Comparison")
plt.xlabel("PR-AUC")
plt.ylabel("Model")
plt.show()

### Observation

Models with stronger PR-AUC and ROC-AUC should be preferred because the dataset is imbalanced.  
PR-AUC is especially important here because it focuses more on the minority `failed` class.

## Step 11: Validation Set Evaluation

Now each model is trained on the training set and evaluated on the validation set.

This helps us choose the final model more clearly.

In [ ]:
validation_results_list = []
trained_models = {}

for model_name, model in models.items():
    current_model = clone(model)
    current_model.fit(X_train, y_train)

    val_prob = get_positive_probability(current_model, X_val)
    metric_row = compute_metrics(y_val, val_prob)
    metric_row["model"] = model_name

    validation_results_list.append(metric_row)
    trained_models[model_name] = current_model

validation_results = pd.DataFrame(validation_results_list).sort_values(["pr_auc", "roc_auc"], ascending=False)
validation_results

In [ ]:
best_model_name = validation_results.iloc[0]["model"]
best_model = trained_models[best_model_name]

print("Best model selected from validation results:", best_model_name)

## Step 12: Threshold Tuning

By default, classification uses a threshold of `0.50`.  
But in bankruptcy prediction, we may want to catch more risky companies, so trying different thresholds is useful.

In [ ]:
val_prob_best = get_positive_probability(best_model, X_val)

threshold_values = np.linspace(0.05, 0.95, 37)
threshold_result_rows = []

for threshold in threshold_values:
    row = compute_metrics(y_val, val_prob_best, threshold=threshold)
    row["threshold"] = threshold
    threshold_result_rows.append(row)

threshold_table = pd.DataFrame(threshold_result_rows)
threshold_table.sort_values(["f1_score", "recall"], ascending=False).head(10)

In [ ]:
best_threshold = threshold_table.sort_values(["f1_score", "recall"], ascending=False).iloc[0]["threshold"]
print(f"Chosen threshold: {best_threshold:.2f}")

plt.figure(figsize=(10, 5))
plt.plot(threshold_table["threshold"], threshold_table["precision"], label="Precision")
plt.plot(threshold_table["threshold"], threshold_table["recall"], label="Recall")
plt.plot(threshold_table["threshold"], threshold_table["f1_score"], label="F1-score")
plt.axvline(best_threshold, linestyle="--", color="black", label=f"Best Threshold = {best_threshold:.2f}")
plt.title("Threshold Tuning on Validation Set")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.show()

## Step 13: Final Training and Test Set Evaluation

After selecting the best model and threshold:

1. Train on `train + validation`
2. Evaluate once on the untouched test set

In [ ]:
final_model = clone(models[best_model_name])
final_model.fit(X_train_val, y_train_val)

test_prob = get_positive_probability(final_model, X_test)

test_results = pd.DataFrame([
    {"setting": "default_threshold_0.50", **compute_metrics(y_test, test_prob, threshold=0.50)},
    {"setting": f"tuned_threshold_{best_threshold:.2f}", **compute_metrics(y_test, test_prob, threshold=best_threshold)},
])
test_results

In [ ]:
test_pred = (test_prob >= best_threshold).astype(int)

print("Classification Report:")
print(classification_report(y_test, test_pred, target_names=["alive", "failed"], zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Pred Alive", "Pred Failed"], yticklabels=["True Alive", "True Failed"])
plt.title(f"Confusion Matrix (Threshold = {best_threshold:.2f})")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_prob)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test, test_prob):.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("ROC Curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].plot(recall_curve, precision_curve, color="#E45756", label=f"PR-AUC = {average_precision_score(y_test, test_prob):.3f}")
axes[1].set_title("Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 14: Feature Importance

To understand which features matter most, we use permutation importance on the final model.

In [ ]:
sample_size = min(len(X_test), 8000)
sample_index = np.random.default_rng(RANDOM_STATE).choice(len(X_test), size=sample_size, replace=False)

perm = permutation_importance(
    final_model,
    X_test.iloc[sample_index],
    y_test.iloc[sample_index],
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
)

importance_df = pd.DataFrame({
    "feature": numeric_features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df.head(15)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(15), x="importance_mean", y="feature", palette="magma")
plt.title("Top 15 Important Features")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.show()

## Step 15: Save the Final Model

This is useful for your future UI, web app, or API.

In [ ]:
artifact = {
    "model_name": best_model_name,
    "pipeline": final_model,
    "threshold": float(best_threshold),
    "feature_columns": numeric_features,
    "target_mapping": target_map,
}

joblib.dump(artifact, "bankruptcy_prediction_artifact.joblib")
print("Saved file: bankruptcy_prediction_artifact.joblib")

## Step 16: Example Prediction for a Future UI

Later, your UI can collect:

- `year`
- `X1` to `X18`

and send those values to the saved model.

In [ ]:
loaded_artifact = joblib.load("bankruptcy_prediction_artifact.joblib")

example_input = X_test.iloc[[0]].copy()
example_probability = loaded_artifact["pipeline"].predict_proba(example_input)[:, 1][0]
example_prediction = int(example_probability >= loaded_artifact["threshold"])

print("Example input row:")
display(example_input)
print(f"Predicted bankruptcy probability: {example_probability:.4f}")
print(f"Predicted class: {example_prediction}  (1 = failed, 0 = alive)")

## Conclusion

In this notebook, we:

1. Loaded and understood the bankruptcy dataset.
2. Performed EDA and studied important patterns.
3. Cleaned and prepared the data for modeling.
4. Used group-aware splitting to avoid data leakage.
5. Trained multiple machine learning models.
6. Compared them using proper classification metrics.
7. Chose a final model and tuned its threshold.
8. Saved the trained model for future deployment.

This notebook is suitable as a beginner-friendly project report as well as a practical ML pipeline.

## Future Scope

You can improve this project further by:

1. finding the real names of features `X1` to `X18`
2. doing hyperparameter tuning
3. using time-based validation
4. calibrating predicted probabilities
5. building a UI that accepts company financial values and returns bankruptcy risk